# Preprocessing i Tekstit dhe Train/Val/Test Split

## 1. Verifikimi i cilësisë së dataset-it final

In [2]:
import pandas as pd

df = pd.read_csv('../data/processed/dataset_final.csv')

print("Shape:", df.shape)
print("\nKolonat:", df.columns.tolist())
print("\nMissing values:\n", df.isnull().sum())

print("\nGjatësia e content (karaktere) - statistika:")
df['content_len'] = df['content'].fillna('').astype(str).apply(len)
print(df['content_len'].describe())

print("\nSa portale kontribuan te dataset-i final:")
print(df['source'].value_counts())

print("\nShpërndarja kategori × portal:")
print(pd.crosstab(df['kategoria_finale'], df['source']))

Shape: (108000, 10)

Kolonat: ['content', 'date', 'title', 'category', 'author', 'source', 'fetched_at', 'category_clean', 'n_labels', 'kategoria_finale']

Missing values:
 content               18
date                   0
title                  0
category               0
author              4034
source                 0
fetched_at             0
category_clean         0
n_labels               0
kategoria_finale       0
dtype: int64

Gjatësia e content (karaktere) - statistika:
count    108000.00000
mean       1349.75412
std        1575.13998
min           0.00000
25%         606.00000
50%         926.00000
75%        1485.00000
max       59475.00000
Name: content_len, dtype: float64

Sa portale kontribuan te dataset-i final:
source
Ballkani        30638
Klankosova      19773
Insajderi       18935
Indeksonline    11742
Telegrafi       11541
GazetaBlic       5353
Bota_al          4761
Lajme_al         4506
JavaNews_al       315
Lajmi_net         277
Kallxo.com        159
Name: count, dty

## 2. Pastrimi i tekstit (HTML entities, whitespace, artikuj shumë të shkurtër)

In [3]:
import re

# 1. hiq rreshtat me content bosh
df = df.dropna(subset=['content']).copy()
print(f"Pas heqjes së content bosh: {df.shape}")

# 2. funksion pastrimi teksti
def pastro_tekst(text):
    text = str(text)
    # HTML entities të zakonshme
    text = text.replace('&amp;', '&').replace('&quot;', '"').replace('&#039;', "'")
    text = text.replace('&nbsp;', ' ').replace('&rsquo;', "'").replace('&ldquo;', '"').replace('&rdquo;', '"')
    # whitespace i tepërt (newline-e, tab-e, hapësira shumëfishta)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['content_pastruar'] = df['content'].apply(pastro_tekst)
df['title_pastruar'] = df['title'].apply(pastro_tekst)

# 3. rillogarit gjatësinë pas pastrimit
df['content_len'] = df['content_pastruar'].apply(len)

print("\nStatistika e gjatësisë PAS pastrimit:")
print(df['content_len'].describe())

# 4. shiko sa artikuj janë shumë të shkurtër (potencialisht problematikë)
print("\nArtikuj me malli se 100 karaktere:", (df['content_len'] < 100).sum())
print("Artikuj me malli se 50 karaktere:", (df['content_len'] < 50).sum())
print("\nShembuj të artikujve shumë të shkurtër (< 50 karaktere):")
print(df[df['content_len'] < 50][['content_pastruar', 'kategoria_finale']].head(10).to_string())

Pas heqjes së content bosh: (107982, 11)

Statistika e gjatësisë PAS pastrimit:
count    107982.00000
mean       1349.97507
std        1575.17194
min          30.00000
25%         606.00000
50%         926.00000
75%        1485.00000
max       59475.00000
Name: content_len, dtype: float64

Artikuj me malli se 100 karaktere: 30
Artikuj me malli se 50 karaktere: 2

Shembuj të artikujve shumë të shkurtër (< 50 karaktere):
                            content_pastruar kategoria_finale
38756  Shikoni këto foto klishe dhe komike …      Kuriozitete
44234         Mundohet ta mbajë, por e kotë…      Kuriozitete


## 3. Filtrimi i artikujve shumë të shkurtër

In [4]:
# hiq artikuj nën 100 karaktere (praktikisht bosh/pa vlerë informative)
MIN_GJATESIA = 100

para = len(df)
df = df[df['content_len'] >= MIN_GJATESIA].copy()
pas = len(df)

print(f"U hoqën {para - pas} artikuj nën {MIN_GJATESIA} karaktere")
print(f"Shape pas filtrimit: {df.shape}")

print("\nShpërndarja finale e kategorive (pas filtrimit):")
print(df['kategoria_finale'].value_counts())

U hoqën 30 artikuj nën 100 karaktere
Shape pas filtrimit: (107952, 13)

Shpërndarja finale e kategorive (pas filtrimit):
kategoria_finale
Shëndet        12000
Politikë       11999
Botë           11998
Showbiz        11998
Sport          11997
Lifestyle      11995
Ekonomi        11994
Kulturë        11987
Kuriozitete    11984
Name: count, dtype: int64


## 4. Krijimi i tekstit final (title + content)

In [5]:
df['text_final'] = df['title_pastruar'] + '. ' + df['content_pastruar']

print("Shembull i tekstit final:")
print(df['text_final'].iloc[0][:300])

print("\n\nStatistika e gjatësisë së tekstit final (karaktere):")
df['text_final_len'] = df['text_final'].apply(len)
print(df['text_final_len'].describe())

Shembull i tekstit final:
Greqia: Nuk do të negociojmë me trojkën. Por, nuk vonon paralajmërimi…. Ministri i ri i Financave i Greqisë thotë se qeveria e tij nuk do të negociojë me trojkën për kushtet e paketës së shpëtimit. Yanis Varoufakis tha se Greqia kërkon bisedime direkte me udhëheqësit e Eurozonës, për t’u përpjekur q


Statistika e gjatësisë së tekstit final (karaktere):
count    107952.000000
mean       1410.340939
std        1576.188827
min         122.000000
25%         666.000000
50%         988.000000
75%        1550.000000
max       59564.000000
Name: text_final_len, dtype: float64


## 5. Ruajtja e dataset-it të pastruar

In [6]:
# mbaj vetëm kolonat e nevojshme për trajnim
df_clean = df[['text_final', 'kategoria_finale', 'source', 'date']].copy()
df_clean = df_clean.rename(columns={'kategoria_finale': 'label'})

df_clean.to_csv('../data/processed/dataset_clean.csv', index=False)
print(f"U ruajt: data/processed/dataset_clean.csv ({df_clean.shape[0]} rreshta)")
print("\nKolonat finale:", df_clean.columns.tolist())
df_clean.head()

U ruajt: data/processed/dataset_clean.csv (107952 rreshta)

Kolonat finale: ['text_final', 'label', 'source', 'date']


,text_final,label,source,date
0,"Greqia: Nuk do të negociojmë me trojkën. Por, ...",Botë,Bota_al,2015-01-31T00:36:57
1,"Rikthehen tmerret, trupat dergjen shtrirë rrug...",Botë,Bota_al,2015-01-31T00:52:06
2,"“Nuk duam luftë me Izraelin, por nuk i druhemi...",Botë,Bota_al,2015-01-31T00:57:16
3,Kuba hap dyert për Coca Cola dhe McDonald’s. P...,Botë,Bota_al,2015-01-31T01:06:29
4,Shtyhet rrahja me kamzhik e bloggerit saudit. ...,Botë,Bota_al,2015-01-31T01:30:05


## 6. Train / Validation / Test Split (70% / 15% / 15%, stratified)

In [ ]:
from sklearn.model_selection import train_test_split

# hapi 1: nda train (70%) nga temp (30%)
train_df, temp_df = train_test_split(
    df_clean, 
    test_size=0.30, 
    stratify=df_clean['label'], 
    random_state=42
)

# hapi 2: nda temp (30%) në validation (15%) dhe test (15%) - domethënë gjysmë-gjysmë të temp
val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.50, 
    stratify=temp_df['label'], 
    random_state=42
)

print(f"Train: {train_df.shape[0]} rreshta ({train_df.shape[0]/len(df_clean)*100:.1f}%)")
print(f"Validation: {val_df.shape[0]} rreshta ({val_df.shape[0]/len(df_clean)*100:.1f}%)")
print(f"Test: {test_df.shape[0]} rreshta ({test_df.shape[0]/len(df_clean)*100:.1f}%)")

print("\nShpërndarja e kategorive në TRAIN:")
print(train_df['label'].value_counts())

print("\nShpërndarja e kategorive në VALIDATION:")
print(val_df['label'].value_counts())

print("\nShpërndarja e kategorive në TEST:")
print(test_df['label'].value_counts())

# ruaj tre file-t
train_df.to_csv('../data/processed/train.csv', index=False)
val_df.to_csv('../data/processed/val.csv', index=False)
test_df.to_csv('../data/processed/test.csv', index=False)
print("\nU ruajtën: train.csv, val.csv, test.csv te data/processed/")